In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print("All imports successful!")

All imports successful!


In [2]:
# Load all tickers
with open('../data/processed/ticker_list.txt', 'r') as f:
    tickers = [line.strip() for line in f.readlines()]

# Load all data
dfs = []
for ticker in tickers:
    try:
        df = pd.read_csv(f'../data/processed/{ticker}_features.csv')
        dfs.append(df)
    except:
        pass

data = pd.concat(dfs, ignore_index=True)
data['Date'] = pd.to_datetime(data['Date'])
data = data[data['Date'] >= pd.Timestamp('2024-01-01')]

# Feature columns
drop_cols = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume',
             'Trades', 'Turnover', 'Ticker', 'Label', 'Future_Return', 'Sector']
feature_cols = [c for c in data.columns if c not in drop_cols]

print(f"Total rows: {len(data)}")
print(f"Features: {len(feature_cols)}")

Total rows: 139651
Features: 23


In [3]:
SEQ_LEN = 10

def create_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i:i+seq_len])
        ys.append(y[i+seq_len])
    return np.array(Xs), np.array(ys)


class StockLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.3):
        super(StockLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 3)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        return self.classifier(lstm_out[:, -1, :])


def train_lstm_proba(X_train_t, y_train_t, X_test_t, input_dim):
    """Train LSTM and return probabilities"""
    train_dl = DataLoader(
        TensorDataset(X_train_t, y_train_t),
        batch_size=128, shuffle=True
    )
    model = StockLSTM(input_dim=input_dim)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

    patience = 30
    best_acc = 0
    best_weights = None
    no_improve = 0

    for epoch in range(1000):
        model.train()
        for X_batch, y_batch in train_dl:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            _, predicted = torch.max(model(X_test_t), 1)
            acc = accuracy_score(y_test_lstm, predicted.numpy())

        if acc > best_acc:
            best_acc = acc
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            break

    model.load_state_dict(best_weights)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(X_test_t), dim=1).numpy()

    return probs, best_acc


print("Functions ready!")

Functions ready!


In [4]:
target_sectors = [
    'Hydro Power',
    'Microfinance',
    'Commercial Banks',
    'Development Banks',
    'Finance',
    'Life Insurance',
    'Manufacturing And Processing',
    'Non Life Insurance',
]

results_ensemble = {}
scaler = StandardScaler()

for sector in target_sectors:
    print(f"\n{'='*45}")
    print(f"Training: {sector}")

    sector_data = data[data['Sector'] == sector].copy()
    sector_data = sector_data.dropna(subset=feature_cols + ['Label'])

    n_stocks = sector_data['Ticker'].nunique()
    n_rows = len(sector_data)
    print(f"Stocks: {n_stocks} | Rows: {n_rows}")

    if n_rows < 500:
        print(f"Skipping — not enough data")
        continue

    X = sector_data[feature_cols].values
    y = sector_data['Label'].values

    # Scale
    X_scaled = scaler.fit_transform(X)

    # ── XGBoost ──────────────────────────────────────────────
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42, stratify=y
    )

    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='mlogloss',
        verbosity=0
    )
    xgb.fit(X_train, y_train)
    xgb_probs = xgb.predict_proba(X_test)
    xgb_acc = accuracy_score(y_test, xgb.predict(X_test))
    print(f"XGBoost Acc: {xgb_acc*100:.1f}%")

    # ── LSTM ─────────────────────────────────────────────────
    X_seq, y_seq = create_sequences(X_scaled, y, SEQ_LEN)

    X_train_seq, X_test_seq, y_train_seq, y_test_lstm = train_test_split(
        X_seq, y_seq, test_size=0.2, random_state=42, stratify=y_seq
    )

    X_train_t = torch.FloatTensor(X_train_seq)
    y_train_t = torch.LongTensor(y_train_seq)
    X_test_t  = torch.FloatTensor(X_test_seq)

    lstm_probs, lstm_acc = train_lstm_proba(X_train_t, y_train_t, X_test_t, X_seq.shape[2])
    print(f"LSTM Acc: {lstm_acc*100:.1f}%")

    # ── Ensemble — average probabilities ─────────────────────
    # Align sizes (sequences reduce rows by SEQ_LEN)
    xgb_probs_aligned = xgb.predict_proba(X_test_seq.reshape(len(X_test_seq), -1)[:, :23])
    ensemble_probs = (xgb_probs_aligned + lstm_probs) / 2
    ensemble_preds = np.argmax(ensemble_probs, axis=1)
    ensemble_acc = accuracy_score(y_test_lstm, ensemble_preds)
    print(f"Ensemble Acc: {ensemble_acc*100:.1f}%")
    print(classification_report(y_test_lstm, ensemble_preds, target_names=['Sell', 'Hold', 'Buy']))

    results_ensemble[sector] = {
        'xgb_acc': round(xgb_acc, 4),
        'lstm_acc': round(lstm_acc, 4),
        'ensemble_acc': round(ensemble_acc, 4),
        'stocks': n_stocks
    }

# ── Summary ──────────────────────────────────────────────────
print("\n" + "="*65)
print("ENSEMBLE SUMMARY (XGBoost + LSTM)")
print("="*65)
for sector, res in results_ensemble.items():
    print(f"{sector:35} | XGB: {res['xgb_acc']*100:.1f}% | LSTM: {res['lstm_acc']*100:.1f}% | Ensemble: {res['ensemble_acc']*100:.1f}%")


Training: Hydro Power
Stocks: 97 | Rows: 52088
XGBoost Acc: 47.3%
LSTM Acc: 49.2%
Ensemble Acc: 47.5%
              precision    recall  f1-score   support

        Sell       0.46      0.38      0.42      3314
        Hold       0.49      0.72      0.58      4268
         Buy       0.45      0.21      0.29      2834

    accuracy                           0.48     10416
   macro avg       0.47      0.44      0.43     10416
weighted avg       0.47      0.48      0.45     10416


Training: Microfinance
Stocks: 50 | Rows: 26978
XGBoost Acc: 50.1%
LSTM Acc: 51.5%
Ensemble Acc: 49.3%
              precision    recall  f1-score   support

        Sell       0.42      0.25      0.32      1540
        Hold       0.52      0.79      0.63      2497
         Buy       0.43      0.22      0.29      1357

    accuracy                           0.49      5394
   macro avg       0.46      0.42      0.41      5394
weighted avg       0.47      0.49      0.45      5394


Training: Commercial Banks
Sto